# FRED Time Series — MCP & DataFetcherAgent Tests

Tests the MCP interface directly and via the `DataFetcherAgent`.

In [ ]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('../..'))
sys.path.insert(0, os.path.abspath('../../..'))

from apps.agentic.core.utils import set_anthropic_env
set_anthropic_env()

## 1. Direct MCP Tool Call

Verify the MCP server is reachable and `fred.series_observations` returns expected data.

In [ ]:
from lib.mcp_client import MCPClient, MCPClientConfig
from apps.agentic.core.constants import MCP_URL

mcp_config = MCPClientConfig(url=MCP_URL)

async def call_tool(tool_name, arguments=None):
    async with MCPClient(mcp_config) as client:
        return await client.call_tool(tool_name, arguments or {})

async def list_mcp_tools():
    async with MCPClient(mcp_config) as client:
        return await client.list_tools()

tools = await list_mcp_tools()
for t in tools:
    print(f'{t.name}: {t.description}')

In [ ]:
SERIES_ID = 'BOGMBBM'

result = await call_tool('fred.series_observations', {'series_id': SERIES_ID, 'limit': 5})
data = result.structuredContent['result']

print(f"count  : {data['count']}")
print(f"limit  : {data['limit']}")
print(f"offset : {data['offset']}")
print()
for obs in data['observations']:
    print(obs)

In [ ]:
from datetime import datetime

result = await call_tool('fred.series_observations', {'series_id': SERIES_ID, 'limit': 10000})
observations = result.structuredContent['result']['observations']

timestamps = []
values = []
for obs in observations:
    if obs['value'] == '.':
        continue
    timestamps.append(datetime.strptime(obs['date'], '%Y-%m-%d'))
    values.append(float(obs['value']))

print(f'Series  : {SERIES_ID}')
print(f'Points  : {len(timestamps)}')
print(f'Start   : {timestamps[0].date()}')
print(f'End     : {timestamps[-1].date()}')
print(f'Min val : {min(values):.4f}')
print(f'Max val : {max(values):.4f}')

## 2. MCP Tool Discovery via `MultiServerMCPClient`

Verify that `langchain-mcp-adapters` discovers tools from the MCP server correctly.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({'fred': {'transport': 'sse', 'url': MCP_URL}})
discovered_tools = await client.get_tools()

print(f'Discovered {len(discovered_tools)} tools:')
for t in discovered_tools:
    print(f'  {t.name}: {t.description}')

## 3. DataFetcherAgent — Direct Invocation

Create the agent via `DataFetcherAgent.create()` and invoke it directly.

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
import shortuuid

from apps.agentic.agents.data.data_fetcher_agent import DataFetcherAgent

agent = await DataFetcherAgent.create()
print(f'Agent initialised with tools: {list(agent.tool_node.tools_by_name.keys())}')

In [ ]:
state = {'messages': [HumanMessage(content=f'Fetch the {SERIES_ID} series from FRED with a limit of 10000.')]}
run_config = RunnableConfig(configurable={'thread_id': shortuuid.uuid()})

result = await agent.agent.ainvoke(state, run_config)
response = result['messages'][-1].content
print(response[:500])

## 4. Plot the Fetched Series

In [ ]:
import numpy
import matplotlib
matplotlib.use('Agg')
from matplotlib import pyplot
from IPython.display import Image
from lib import config as plot_config
from lib.plots import curve
from lib.utils import generate_plot_file_name

pyplot.style.use(plot_config.glyfish_style)

uuid = shortuuid.uuid()
output_file = generate_plot_file_name('fred_time_series', path='./html/plots', uuid=uuid)

curve(numpy.array(values), numpy.array(timestamps),
      title=SERIES_ID, xlabel='Date', ylabel='Value',
      figsize=(12, 5), file_name=output_file)

Image(output_file)